# Dynamic Value References — Lakeflow Jobs

> **DataMindAI | Ahmed**  
> Using `{{ }}` placeholders to pass job metadata and task outputs between tasks  
> **Student data pipeline:** Ingest → Transform → Dashboard

---
## What is a Dynamic Value Reference?
A `{{ }}` placeholder in a **job or task configuration field** that Databricks resolves
to a real string before the notebook starts.

| Reference | Resolves to |
|-----------|------------|
| `{{job.start_time.iso_date}}` | `2024-01-15` |
| `{{job.run_id}}` | `67890` |
| `{{job.id}}` | `12345` |
| `{{tasks.ingest_students.values.record_count}}` | `10` |
| `{{tasks.transform_scores.values.pass_rate}}` | `70.0` |
| `{{tasks.find_at_risk_sql.output.rows}}` | JSON array of rows |
| `{{input.student_id}}` | `S009` (inside For-Each) |

> **Rule:** `{{ }}` lives in the job/task config — never inside a notebook.
> The notebook receives the resolved string via `dbutils.widgets.get()`.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
spark.sql('CREATE CATALOG IF NOT EXISTS demo_catalog')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.gold')
print('Setup complete.')

---
## TASK 1 — ingest_students
---

## Task 1 — ingest_students

**Job parameter wiring:**
```
Job-level parameter:  run_date  =  {{job.start_time.iso_date}}
Task parameter:       run_date  =  {{job.parameters.run_date}}
Task parameter:       run_id    =  {{job.parameters.run_id}}
```

The notebook reads these with `dbutils.widgets.get()` and then publishes
three task values for downstream tasks to consume.

In [ ]:
# ── ingest_students.py ──────────────────────────────────────────────────
# RECEIVES via widget parameters:
#   run_date  <--  {{job.start_time.iso_date}}
#   run_id    <--  {{job.run_id}}
# PUBLISHES task values:
#   record_count, run_date, at_risk_ingested

from pyspark.sql import Row
from pyspark.sql.functions import lit

# In a job: these come from {{ }} references.
# For local testing: set defaults here.
try:
    run_date = dbutils.widgets.get('run_date')
    run_id   = dbutils.widgets.get('run_id')
except:
    run_date = '2024-01-15'   # local fallback
    run_id   = 'LOCAL'

print(f'Run ID: {run_id}  |  Run date: {run_date}')

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94),
]
df = (spark.createDataFrame(students)
        .withColumn('assessment_date', lit(run_date))
        .withColumn('job_run_id', lit(run_id)))

df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.student_assessments')

record_count     = df.count()
at_risk_ingested = df.filter('score < 40 OR attendance < 50').count()

dbutils.jobs.taskValues.set(key='record_count',     value=int(record_count))
dbutils.jobs.taskValues.set(key='run_date',         value=run_date)
dbutils.jobs.taskValues.set(key='at_risk_ingested', value=int(at_risk_ingested))

print(f'Ingested {record_count} students  |  At-risk: {at_risk_ingested}')
df.show()

---
## TASK 2 — transform_scores
---

## Task 2 — transform_scores

**Task parameter wiring in the Job UI:**
```
record_count  =  {{tasks.ingest_students.values.record_count}}
run_date      =  {{tasks.ingest_students.values.run_date}}
at_risk_in    =  {{tasks.ingest_students.values.at_risk_ingested}}
```

In [ ]:
# ── transform_scores.py ─────────────────────────────────────────────────
# RECEIVES (via widget params resolved from Task 1 task values):
#   record_count  <--  {{tasks.ingest_students.values.record_count}}
#   run_date      <--  {{tasks.ingest_students.values.run_date}}
#   at_risk_in    <--  {{tasks.ingest_students.values.at_risk_ingested}}
# PUBLISHES: pass_rate, avg_score, at_risk_count, top_performer

from pyspark.sql.functions import when, col, avg as spark_avg

try:
    record_count = int(dbutils.widgets.get('record_count'))
    run_date     = dbutils.widgets.get('run_date')
    at_risk_in   = int(dbutils.widgets.get('at_risk_in'))
except:
    record_count, run_date, at_risk_in = 10, '2024-01-15', 2

print(f'Transform for {run_date}  |  {record_count} records  |  {at_risk_in} at-risk at ingest')

if record_count == 0:
    dbutils.notebook.exit('SKIP: empty ingest')

df = spark.table('demo_catalog.bronze.student_assessments')

df_graded = df.withColumn('grade',
    when(col('score') >= 70, 'Distinction')
    .when(col('score') >= 55, 'Merit')
    .when(col('score') >= 40, 'Pass')
    .otherwise('Fail')
).withColumn('at_risk',
    when((col('score') < 40) | (col('attendance') < 50), True).otherwise(False)
)
df_graded.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.silver.student_grades')

total      = df_graded.count()
passed     = df_graded.filter("grade != 'Fail'").count()
at_risk_ct = df_graded.filter('at_risk = true').count()
avg_sc     = round(df_graded.agg(spark_avg('score')).collect()[0][0], 1)
pass_rate  = round((passed / total) * 100, 1)
top_row    = df_graded.orderBy(col('score').desc()).first()
top_str    = f"{top_row['name']} ({top_row['score']}% - {top_row['grade']})"

dbutils.jobs.taskValues.set(key='pass_rate',     value=float(pass_rate))
dbutils.jobs.taskValues.set(key='avg_score',     value=float(avg_sc))
dbutils.jobs.taskValues.set(key='at_risk_count', value=int(at_risk_ct))
dbutils.jobs.taskValues.set(key='top_performer', value=top_str)
dbutils.jobs.taskValues.set(key='run_date',      value=run_date)

print(f'Pass rate: {pass_rate}%  |  Avg: {avg_sc}  |  At-risk: {at_risk_ct}  |  Top: {top_str}')
df_graded.show()

---
## TASK 3 — dashboard_refresh
---

## Task 3 — dashboard_refresh

**Task parameter wiring — reads from BOTH Task 1 and Task 2:**
```
pass_rate      =  {{tasks.transform_scores.values.pass_rate}}
at_risk_count  =  {{tasks.transform_scores.values.at_risk_count}}
top_performer  =  {{tasks.transform_scores.values.top_performer}}
avg_score      =  {{tasks.transform_scores.values.avg_score}}
run_date       =  {{tasks.transform_scores.values.run_date}}
record_count   =  {{tasks.ingest_students.values.record_count}}
job_id         =  {{job.id}}
run_id         =  {{job.run_id}}
```

In [ ]:
# ── dashboard_refresh.py ────────────────────────────────────────────────
try:
    pass_rate     = float(dbutils.widgets.get('pass_rate'))
    at_risk_count = int(dbutils.widgets.get('at_risk_count'))
    top_performer = dbutils.widgets.get('top_performer')
    avg_score     = float(dbutils.widgets.get('avg_score'))
    run_date      = dbutils.widgets.get('run_date')
    record_count  = int(dbutils.widgets.get('record_count'))
    job_id        = dbutils.widgets.get('job_id')
    run_id        = dbutils.widgets.get('run_id')
except:
    pass_rate, at_risk_count = 70.0, 2
    top_performer = 'Sophie Williams (91% - Distinction)'
    avg_score, run_date, record_count = 63.1, '2024-01-15', 10
    job_id, run_id = 'LOCAL', 'LOCAL'

status = 'ON TRACK' if pass_rate >= 70 else ('NEEDS ATTENTION' if pass_rate >= 50 else 'CRITICAL')

print('=' * 58)
print('   STUDENT ASSESSMENT DASHBOARD')
print(f'   Date: {run_date}  |  Students: {record_count}  |  Job: {job_id}  |  Run: {run_id}')
print('=' * 58)
print(f'   Pass Rate      :  {pass_rate}%  --  {status}')
print(f'   Average Score  :  {avg_score}%')
print(f'   At-Risk        :  {at_risk_count} student(s)')
print(f'   Top Performer  :  {top_performer}')

if at_risk_count > 0:
    print()
    print('-' * 58)
    print(f'   AT-RISK STUDENTS ({at_risk_count}):')
    df = spark.table('demo_catalog.silver.student_grades')
    for r in df.filter('at_risk = true').orderBy('score').collect():
        print(f'   {r["name"]:22}  Score:{r["score"]}%  Att:{r["attendance"]}%  {r["grade"]}')

print('=' * 58)

from pyspark.sql import Row
spark.createDataFrame([Row(
    run_date=run_date, run_id=run_id, job_id=job_id,
    pass_rate=pass_rate, avg_score=avg_score,
    at_risk_count=at_risk_count, top_performer=top_performer
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.dashboard_history')

---
## BONUS — SQL Output Rows + For-Each
---

## Bonus — SQL Output Rows → For-Each Per-Student Alert

**SQL task: `find_at_risk_sql`**
```sql
SELECT student_id, name, score, attendance, grade,
    CASE
        WHEN score < 30 THEN 'URGENT: score critically low'
        WHEN attendance < 50 THEN 'URGENT: attendance below 50%'
        ELSE 'MONITOR: borderline'
    END AS alert_level
FROM demo_catalog.silver.student_grades
WHERE at_risk = true
ORDER BY score ASC
```

**For-Each task:**
- Inputs: `{{tasks.find_at_risk_sql.output.rows}}`
- Concurrency: 3

**Nested task parameters:**
```
student_id   =  {{input.student_id}}
student_name =  {{input.name}}
score        =  {{input.score}}
alert_level  =  {{input.alert_level}}
```

In [ ]:
# ── send_student_alert.py (nested For-Each task) ────────────────────────
# Runs ONCE PER ROW from the SQL task output
import datetime

try:
    student_id   = dbutils.widgets.get('student_id')
    student_name = dbutils.widgets.get('student_name')
    score        = int(dbutils.widgets.get('score'))
    alert_level  = dbutils.widgets.get('alert_level')
except:
    student_id, student_name, score = 'S009', 'Tariq Ahmed', 29
    alert_level = 'URGENT: score critically low'

print(f'Alert for: {student_name} ({student_id})  Score: {score}%')
print(f'Level: {alert_level}')

from pyspark.sql import Row
spark.createDataFrame([Row(
    student_id=student_id, student_name=student_name,
    score=score, alert_level=alert_level,
    alerted_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.student_alerts_log')

print(f'Alert logged for {student_name}.')